In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

shipments_df = pd.read_csv("../data/drug_shipments_200.csv")
print(f"Data shape: {shipments_df.shape}")

In [ ]:
with open("../data/drug_shipments_200_meta.md", "r") as f:
    metadata_content = f.read()

display(Markdown(metadata_content))

In [ ]:
shipments_df.describe()

### Phase 1: Data Quality & Overview ---

In [ ]:
# Date parsing
shipments_df["SHIPPED_DT"] = pd.to_datetime(shipments_df["SHIPPED_DT"])
shipments_df["LOAD_DT"] = pd.to_datetime(shipments_df["LOAD_DT"])

# Derived time columns
shipments_df["SHIP_MONTH"] = shipments_df["SHIPPED_DT"].dt.to_period("M")
shipments_df["SHIP_WEEK"] = shipments_df["SHIPPED_DT"].dt.isocalendar().week.astype(int)
shipments_df["SHIP_DOW"] = shipments_df["SHIPPED_DT"].dt.day_name()
shipments_df["LOAD_LAG_DAYS"] = (
    shipments_df["LOAD_DT"] - shipments_df["SHIPPED_DT"]
).dt.days

print("=== Data Types ===")
print(shipments_df.dtypes.to_string())
print(f"\n=== Shape: {shipments_df.shape} ===")
shipments_df.head()

In [ ]:
# --- Missing Values & Duplicates ---

# Missing values
null_counts = shipments_df.isnull().sum()
null_pct = (null_counts / len(shipments_df) * 100).round(2)
null_summary = pd.DataFrame({"Null Count": null_counts, "Null %": null_pct})
null_summary = null_summary[null_summary["Null Count"] > 0]
if null_summary.empty:
    print("No missing values found in any column.")
else:
    display(null_summary)

# Missing values heatmap (shows pattern even if all complete)
fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(shipments_df.isnull().T, cbar=True, cmap="YlOrRd", yticklabels=True, ax=ax)
ax.set_title("Missing Values Heatmap (yellow = missing)")
plt.tight_layout()
plt.show()

# Duplicates
dup_rows = shipments_df.duplicated().sum()
dup_patients = shipments_df["PATIENT_KEY"].nunique()
print(f"\nDuplicate rows: {dup_rows}")
print(f"Unique patients: {dup_patients} out of {len(shipments_df)} records")
print(f"Avg shipments per patient: {len(shipments_df) / dup_patients:.2f}")

# Uniqueness of key ID columns
id_cols = [
    "PATIENT_KEY",
    "PBR_KEY",
    "PBR_NPI",
    "PLN_KEY",
    "PRI_PAYR_KEY",
    "PAYR_KEY",
    "AFFIL_NBR",
    "NDC",
]
id_unique = pd.DataFrame(
    {
        "Column": id_cols,
        "Unique Values": [shipments_df[c].nunique() for c in id_cols],
        "Total Rows": len(shipments_df),
    }
)
display(id_unique)

In [ ]:
# --- Data Type Audit & Anomaly Detection ---

numeric_cols = [
    "GAP_DAYS",
    "ADJ_DAYS_SPPL",
    "SHIP_QTY",
    "TAT",
    "ASST_OBTAIN_VAL",
    "RX30_DAYS_SUPPLY",
    "SMF_FLAG_ID",
    "HUB_IND_ID",
    "HAS_FOUNDATION_GRANT_KW",
    "HAS_COPAY_KW",
    "NET_PATIENT_OOP_RANGE_ID",
    "PRI_PATIENT_OOP_RANGE_ID",
]

audit = shipments_df[numeric_cols].agg(["min", "max", "mean", "median"]).T
audit.columns = ["Min", "Max", "Mean", "Median"]
display(Markdown("**Numeric Column Ranges**"))
display(audit)

# Flag anomalies
anomalies = []
if (shipments_df["GAP_DAYS"] < 0).any():
    anomalies.append(
        f"  - Negative GAP_DAYS: {(shipments_df['GAP_DAYS'] < 0).sum()} rows"
    )
if (shipments_df["TAT"] < 0).any():
    anomalies.append(f"  - Negative TAT: {(shipments_df['TAT'] < 0).sum()} rows")
if (shipments_df["SHIP_QTY"] <= 0).any():
    anomalies.append(
        f"  - Zero/negative SHIP_QTY: {(shipments_df['SHIP_QTY'] <= 0).sum()} rows"
    )
if (shipments_df["ADJ_DAYS_SPPL"] <= 0).any():
    anomalies.append(
        f"  - Zero/negative ADJ_DAYS_SPPL: {(shipments_df['ADJ_DAYS_SPPL'] <= 0).sum()} rows"
    )

tat_p95 = shipments_df["TAT"].quantile(0.95)
tat_outliers = (shipments_df["TAT"] > tat_p95).sum()
anomalies.append(f"  - TAT > 95th pctl ({tat_p95:.1f} days): {tat_outliers} rows")

gap_p95 = shipments_df["GAP_DAYS"].quantile(0.95)
gap_outliers = (shipments_df["GAP_DAYS"] > gap_p95).sum()
anomalies.append(f"  - GAP_DAYS > 95th pctl ({gap_p95:.0f} days): {gap_outliers} rows")

print("Anomaly Check:")
if anomalies:
    print("\n".join(anomalies))
else:
    print("  No anomalies detected.")

### Phase 2: Univariate Analysis — Numeric Columns

Distribution analysis for continuous and binary numeric columns including skewness and kurtosis metrics.

In [ ]:
# --- Enhanced Descriptive Statistics ---

continuous_cols = ["TAT", "GAP_DAYS", "ADJ_DAYS_SPPL", "SHIP_QTY", "ASST_OBTAIN_VAL"]

desc = shipments_df[continuous_cols].describe().T
desc["skewness"] = shipments_df[continuous_cols].skew()
desc["kurtosis"] = shipments_df[continuous_cols].kurtosis()
desc["iqr"] = desc["75%"] - desc["25%"]
display(Markdown("**Enhanced Descriptive Statistics (continuous numeric columns)**"))
display(desc.round(3))

In [ ]:
# --- Distribution Plots: Histograms + KDE ---

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    ax = axes[i]
    shipments_df[col].hist(bins=25, ax=ax, alpha=0.6, edgecolor="black", density=True)
    shipments_df[col].plot.kde(ax=ax, color="crimson", linewidth=2)
    ax.set_title(f"{col}", fontsize=12, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    median_val = shipments_df[col].median()
    ax.axvline(
        median_val,
        color="navy",
        linestyle="--",
        linewidth=1.5,
        label=f"Median={median_val:.1f}",
    )
    ax.legend(fontsize=8)

axes[-1].set_visible(False)  # hide unused 6th subplot
fig.suptitle(
    "Distributions of Continuous Numeric Columns",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# --- Boxplots for Continuous Numerics ---

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for i, col in enumerate(continuous_cols):
    sns.boxplot(
        y=shipments_df[col], ax=axes[i], color=sns.color_palette("muted")[i], width=0.5
    )
    axes[i].set_title(col, fontsize=11, fontweight="bold")
    axes[i].set_ylabel("")

fig.suptitle(
    "Boxplots — Outlier Detection for Continuous Columns",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

# --- Binary / Flag Column Value Counts ---
flag_cols = [
    "RX30_DAYS_SUPPLY",
    "HAS_FOUNDATION_GRANT_KW",
    "HAS_COPAY_KW",
    "HUB_IND_ID",
    "SMF_FLAG_ID",
]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, col in enumerate(flag_cols):
    vc = shipments_df[col].value_counts().sort_index()
    vc.plot.bar(ax=axes[i], color=sns.color_palette("Set2")[i], edgecolor="black")
    axes[i].set_title(col, fontsize=10, fontweight="bold")
    axes[i].set_ylabel("Count")
    axes[i].set_xlabel("")
    for j, v in enumerate(vc):
        axes[i].text(j, v + 1, str(v), ha="center", fontsize=9)
    axes[i].tick_params(axis="x", rotation=0)

fig.suptitle(
    "Binary / Flag Column Distributions", fontsize=14, fontweight="bold", y=1.04
)
plt.tight_layout()
plt.show()

## Phase 3: Univariate Analysis — Categorical Columns

Top-N frequency distributions for drug, therapeutic class, payer, geographic, diagnosis, and prescriber dimensions.

In [ ]:
# --- Patient Type & ICD-10 Diagnosis Codes ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Patient Type (N/R)
vc_pt = shipments_df["PATIENT_TYPE"].value_counts()
vc_pt.plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452"], edgecolor="black")
axes[0].set_title("Patient Type (New vs Refill)", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
for j, v in enumerate(vc_pt):
    axes[0].text(
        j, v + 1, f"{v} ({v/len(shipments_df)*100:.1f}%)", ha="center", fontsize=9
    )
axes[0].tick_params(axis="x", rotation=0)

# Top ICD-10 codes
vc_icd = shipments_df["PRI_ICD_10_CD"].value_counts().head(10)
vc_icd.plot.barh(
    ax=axes[1], color=sns.color_palette("viridis", len(vc_icd)), edgecolor="black"
)
axes[1].set_title("Top 10 ICD-10 Diagnosis Codes", fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# --- Drug & Therapeutic Class Distribution ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Therapeutic class
vc_tc = shipments_df["THRPC_CLASS"].value_counts()
vc_tc.plot.barh(
    ax=axes[0], color=sns.color_palette("Set2", len(vc_tc)), edgecolor="black"
)
axes[0].set_title("Therapeutic Class", fontweight="bold")
axes[0].set_xlabel("Count")
axes[0].invert_yaxis()

# Therapeutic subclass
vc_sub = shipments_df["TC_SUB_CLASS"].value_counts()
vc_sub.plot.barh(
    ax=axes[1], color=sns.color_palette("Paired", len(vc_sub)), edgecolor="black"
)
axes[1].set_title("Therapeutic Subclass", fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()

# Manufacturer
vc_mfr = shipments_df["MANUFCTR"].value_counts()
vc_mfr.plot.barh(
    ax=axes[2], color=sns.color_palette("husl", len(vc_mfr)), edgecolor="black"
)
axes[2].set_title("Manufacturer", fontweight="bold")
axes[2].set_xlabel("Count")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

# Therapeutic class × subclass crosstab
ct_drug = pd.crosstab(shipments_df["THRPC_CLASS"], shipments_df["TC_SUB_CLASS"])
display(Markdown("**Therapeutic Class × Subclass Crosstab**"))
display(ct_drug)

In [ ]:
# --- Payer Distribution ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Payer type
vc_ptype = shipments_df["PAYR_TYPE"].value_counts()
vc_ptype.plot.pie(
    ax=axes[0],
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("pastel"),
    textprops={"fontsize": 10},
)
axes[0].set_title("Payer Type Distribution", fontweight="bold")
axes[0].set_ylabel("")

# Payer name
vc_pnm = shipments_df["PAYR_NM"].value_counts()
vc_pnm.plot.barh(
    ax=axes[1], color=sns.color_palette("muted", len(vc_pnm)), edgecolor="black"
)
axes[1].set_title("Payer Name Distribution", fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()
for j, v in enumerate(vc_pnm):
    axes[1].text(v + 0.5, j, f"{v}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- Geographic Distribution ---

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Patient state
vc_pst = shipments_df["PATIENT_ST_CD"].value_counts()
vc_pst.plot.bar(
    ax=axes[0], color=sns.color_palette("coolwarm", len(vc_pst)), edgecolor="black"
)
axes[0].set_title("Patient State (PATIENT_ST_CD)", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)

# Prescriber state
vc_states = shipments_df["STATES"].value_counts()
vc_states.plot.bar(
    ax=axes[1], color=sns.color_palette("coolwarm", len(vc_states)), edgecolor="black"
)
axes[1].set_title("Prescriber State (STATES)", fontweight="bold")
axes[1].set_ylabel("Count")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# State alignment check
print(f"Unique patient states: {shipments_df['PATIENT_ST_CD'].nunique()}")
print(f"Unique prescriber states: {shipments_df['STATES'].nunique()}")

In [ ]:
# --- Prescriber Concentration ---

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 15 prescribers by shipment count
top_pbr = shipments_df["PBR_NM"].value_counts().head(15)
top_pbr.plot.barh(
    ax=axes[0], color=sns.color_palette("crest", len(top_pbr)), edgecolor="black"
)
axes[0].set_title("Top 15 Prescribers by Shipment Count", fontweight="bold")
axes[0].set_xlabel("Shipment Count")
axes[0].invert_yaxis()

# Shipments per prescriber distribution
shipments_per_pbr = shipments_df.groupby("PBR_KEY").size()
shipments_per_pbr.hist(
    bins=20, ax=axes[1], color="#4C72B0", edgecolor="black", alpha=0.7
)
axes[1].set_title("Distribution: Shipments per Prescriber", fontweight="bold")
axes[1].set_xlabel("Number of Shipments")
axes[1].set_ylabel("Number of Prescribers")
axes[1].axvline(
    shipments_per_pbr.median(),
    color="red",
    linestyle="--",
    label=f"Median={shipments_per_pbr.median():.0f}",
)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Total unique prescribers: {shipments_df['PBR_KEY'].nunique()}")
print(
    f"Top 5 prescribers account for {top_pbr.head(5).sum()} / {len(shipments_df)} shipments "
    f"({top_pbr.head(5).sum()/len(shipments_df)*100:.1f}%)"
)

In [ ]:
# --- OOP Range Distributions ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(["PRI_PATIENT_OOP_RANGE_ID", "NET_PATIENT_OOP_RANGE_ID"]):
    vc = shipments_df[col].value_counts().sort_index()
    vc.plot.bar(
        ax=axes[i], color=sns.color_palette("YlOrRd", len(vc)), edgecolor="black"
    )
    axes[i].set_title(f"{col}", fontweight="bold")
    axes[i].set_ylabel("Count")
    axes[i].set_xlabel("Range ID (bucket)")
    for j, v in enumerate(vc):
        axes[i].text(j, v + 0.5, str(v), ha="center", fontsize=9)
    axes[i].tick_params(axis="x", rotation=0)

fig.suptitle(
    "Out-of-Pocket Cost Range Distributions (Pre- and Post-Assistance)",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

# NDC / Drug product distribution
vc_ndc = shipments_df["NDC_DISPLAY"].value_counts()
display(Markdown("**Drug Product (NDC) Distribution**"))
display(vc_ndc.to_frame("Count"))

## Phase 4: Temporal Analysis

Shipment volume trends, TAT evolution, load lag distribution, and day-of-week patterns over the observation window.

In [ ]:
# --- Shipment Volume Over Time ---

monthly = shipments_df.groupby("SHIP_MONTH").size()
monthly_by_type = (
    shipments_df.groupby(["SHIP_MONTH", "PATIENT_TYPE"]).size().unstack(fill_value=0)
)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Total monthly volume
ax = axes[0, 0]
monthly.plot(ax=ax, marker="o", color="#4C72B0", linewidth=2)
ax.set_title("Monthly Shipment Volume", fontweight="bold")
ax.set_ylabel("Shipment Count")
ax.set_xlabel("Month")
ax.tick_params(axis="x", rotation=45)

# New vs Refill monthly
ax = axes[0, 1]
monthly_by_type.plot(ax=ax, marker="o", linewidth=2)
ax.set_title("Monthly Volume: New vs Refill", fontweight="bold")
ax.set_ylabel("Shipment Count")
ax.set_xlabel("Month")
ax.legend(title="Patient Type")
ax.tick_params(axis="x", rotation=45)

# TAT trend over time
ax = axes[1, 0]
tat_monthly = shipments_df.groupby("SHIP_MONTH")["TAT"].agg(["mean", "median"])
tat_monthly["mean"].plot(ax=ax, marker="o", label="Mean TAT", linewidth=2)
tat_monthly["median"].plot(
    ax=ax, marker="s", label="Median TAT", linewidth=2, linestyle="--"
)
ax.set_title("TAT Trend Over Time", fontweight="bold")
ax.set_ylabel("TAT (days)")
ax.set_xlabel("Month")
ax.legend()
ax.tick_params(axis="x", rotation=45)

# Day-of-week pattern
ax = axes[1, 1]
dow_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
dow_counts = shipments_df["SHIP_DOW"].value_counts().reindex(dow_order, fill_value=0)
dow_counts.plot.bar(ax=ax, color=sns.color_palette("muted", 7), edgecolor="black")
ax.set_title("Shipments by Day of Week", fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# --- Load Lag Analysis ---

fig, ax = plt.subplots(figsize=(10, 4))
shipments_df["LOAD_LAG_DAYS"].hist(
    bins=30, ax=ax, color="#55A868", edgecolor="black", alpha=0.7
)
ax.axvline(
    shipments_df["LOAD_LAG_DAYS"].median(),
    color="red",
    linestyle="--",
    label=f"Median={shipments_df['LOAD_LAG_DAYS'].median():.0f} days",
)
ax.set_title("Load Lag Distribution (LOAD_DT − SHIPPED_DT)", fontweight="bold")
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Load lag stats:\n{shipments_df['LOAD_LAG_DAYS'].describe().round(2)}")
print(
    f"\nDate range — SHIPPED_DT: {shipments_df['SHIPPED_DT'].min().date()} to {shipments_df['SHIPPED_DT'].max().date()}"
)
print(
    f"Date range — LOAD_DT:    {shipments_df['LOAD_DT'].min().date()} to {shipments_df['LOAD_DT'].max().date()}"
)

## Phase 5: Bivariate & Cross-tabulation Analysis

Correlation structure, segment-level comparisons of key metrics, financial assistance patterns, and out-of-pocket cost shifts.

In [ ]:
# --- Correlation Heatmap ---

corr_cols = [
    "TAT",
    "GAP_DAYS",
    "ADJ_DAYS_SPPL",
    "SHIP_QTY",
    "ASST_OBTAIN_VAL",
    "RX30_DAYS_SUPPLY",
    "HUB_IND_ID",
    "SMF_FLAG_ID",
    "HAS_FOUNDATION_GRANT_KW",
    "HAS_COPAY_KW",
    "NET_PATIENT_OOP_RANGE_ID",
    "PRI_PATIENT_OOP_RANGE_ID",
    "LOAD_LAG_DAYS",
]

corr_matrix = shipments_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.8},
)
ax.set_title(
    "Pearson Correlation Heatmap — Numeric Columns", fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
# --- TAT by Key Segments ---

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

segment_cols = ["PATIENT_TYPE", "THRPC_CLASS", "PAYR_TYPE", "HUB_IND_ID", "SMF_FLAG_ID"]
for i, col in enumerate(segment_cols):
    ax = axes.flatten()[i]
    sns.boxplot(data=shipments_df, x=col, y="TAT", ax=ax, palette="Set2")
    ax.set_title(f"TAT by {col}", fontweight="bold", fontsize=11)
    ax.set_xlabel("")

axes.flatten()[-1].set_visible(False)
fig.suptitle(
    "Turnaround Time (TAT) by Key Segments", fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

# Print median TAT per segment
for col in segment_cols:
    print(f"\nMedian TAT by {col}:")
    print(shipments_df.groupby(col)["TAT"].median().round(2).to_string())

In [ ]:
# --- GAP_DAYS by Key Segments ---

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, col in enumerate(segment_cols):
    ax = axes.flatten()[i]
    sns.boxplot(data=shipments_df, x=col, y="GAP_DAYS", ax=ax, palette="Set3")
    ax.set_title(f"GAP_DAYS by {col}", fontweight="bold", fontsize=11)
    ax.set_xlabel("")

axes.flatten()[-1].set_visible(False)
fig.suptitle("Gap Days by Key Segments", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

for col in segment_cols:
    print(f"\nMedian GAP_DAYS by {col}:")
    print(shipments_df.groupby(col)["GAP_DAYS"].median().round(2).to_string())

In [ ]:
# --- Financial Assistance Analysis ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean ASST_OBTAIN_VAL by payer type
asst_payer = shipments_df.groupby("PAYR_TYPE")["ASST_OBTAIN_VAL"].mean()
asst_payer.plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452"], edgecolor="black")
axes[0].set_title("Mean Assistance ($) by Payer Type", fontweight="bold")
axes[0].set_ylabel("Mean ASST_OBTAIN_VAL ($)")
axes[0].tick_params(axis="x", rotation=0)

# Mean ASST_OBTAIN_VAL by therapeutic class
asst_tc = (
    shipments_df.groupby("THRPC_CLASS")["ASST_OBTAIN_VAL"]
    .mean()
    .sort_values(ascending=True)
)
asst_tc.plot.barh(
    ax=axes[1], color=sns.color_palette("Set2", len(asst_tc)), edgecolor="black"
)
axes[1].set_title("Mean Assistance ($) by Therapeutic Class", fontweight="bold")
axes[1].set_xlabel("Mean ASST_OBTAIN_VAL ($)")

# Foundation grant × copay crosstab
ct_assist = pd.crosstab(
    shipments_df["HAS_FOUNDATION_GRANT_KW"], shipments_df["HAS_COPAY_KW"], margins=True
)
ct_assist.index = [f"Grant={x}" for x in ct_assist.index]
ct_assist.columns = [f"Copay={x}" for x in ct_assist.columns]
sns.heatmap(ct_assist, annot=True, fmt="d", cmap="Blues", ax=axes[2], linewidths=1)
axes[2].set_title("Foundation Grant × Copay Assistance", fontweight="bold")

plt.tight_layout()
plt.show()

# Patients with any assistance
has_assist = (shipments_df["ASST_OBTAIN_VAL"] > 0).sum()
print(
    f"Records with financial assistance (ASST_OBTAIN_VAL > 0): {has_assist} "
    f"({has_assist/len(shipments_df)*100:.1f}%)"
)
print(
    f"Mean assistance when > 0: ${shipments_df.loc[shipments_df['ASST_OBTAIN_VAL'] > 0, 'ASST_OBTAIN_VAL'].mean():.2f}"
)

In [ ]:
# --- New vs Refill Deep Dive ---

compare_cols = ["TAT", "GAP_DAYS", "SHIP_QTY", "ASST_OBTAIN_VAL"]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(compare_cols):
    sns.violinplot(
        data=shipments_df,
        x="PATIENT_TYPE",
        y=col,
        ax=axes[i],
        palette={"N": "#4C72B0", "R": "#DD8452"},
        inner="box",
        cut=0,
    )
    axes[i].set_title(f"{col}: New vs Refill", fontweight="bold", fontsize=11)
    axes[i].set_xlabel("")

fig.suptitle(
    "New (N) vs Refill (R) Patient Comparison", fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

# Summary table
nr_summary = (
    shipments_df.groupby("PATIENT_TYPE")[compare_cols]
    .agg(["mean", "median", "std"])
    .round(2)
)
display(Markdown("**New vs Refill — Summary Statistics**"))
display(nr_summary)

In [ ]:
# --- OOP Shift Heatmap (Pre- vs Post-Assistance) ---

ct_oop = pd.crosstab(
    shipments_df["PRI_PATIENT_OOP_RANGE_ID"], shipments_df["NET_PATIENT_OOP_RANGE_ID"]
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(ct_oop, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set_title(
    "OOP Cost Shift: Pre-Assistance (rows) → Post-Assistance (cols)", fontweight="bold"
)
ax.set_xlabel("NET_PATIENT_OOP_RANGE_ID (post-assistance)")
ax.set_ylabel("PRI_PATIENT_OOP_RANGE_ID (pre-assistance)")
plt.tight_layout()
plt.show()

# Shift direction summary
shipments_df["OOP_SHIFT"] = (
    shipments_df["PRI_PATIENT_OOP_RANGE_ID"] - shipments_df["NET_PATIENT_OOP_RANGE_ID"]
)
shift_counts = (
    shipments_df["OOP_SHIFT"]
    .apply(
        lambda x: (
            "Improved (lower OOP)"
            if x > 0
            else ("No change" if x == 0 else "Worsened (higher OOP)")
        )
    )
    .value_counts()
)
display(Markdown("**OOP Shift Direction**"))
display(shift_counts.to_frame("Count"))

## Phase 6: Shipment Operations Deep Dive

In-depth analysis of turnaround time, gap days / persistence, fulfillment flags, hub processing, 30-day supply compliance, and adjusted days of supply.

In [ ]:
# --- TAT Deep Dive: Distribution, Percentiles & Outlier Profiling ---

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Detailed histogram
ax = axes[0]
shipments_df["TAT"].hist(
    bins=30, ax=ax, color="#4C72B0", edgecolor="black", alpha=0.7, density=True
)
shipments_df["TAT"].plot.kde(ax=ax, color="crimson", linewidth=2)
p95 = shipments_df["TAT"].quantile(0.95)
p99 = shipments_df["TAT"].quantile(0.99)
ax.axvline(p95, color="orange", linestyle="--", linewidth=2, label=f"P95 = {p95:.2f}")
ax.axvline(p99, color="red", linestyle="--", linewidth=2, label=f"P99 = {p99:.2f}")
ax.axvline(
    shipments_df["TAT"].median(),
    color="navy",
    linestyle="-.",
    linewidth=1.5,
    label=f"Median = {shipments_df['TAT'].median():.2f}",
)
ax.set_title("TAT Distribution with Percentile Markers", fontweight="bold")
ax.set_xlabel("TAT (days)")
ax.legend(fontsize=9)

# TAT by drug
ax = axes[1]
sns.boxplot(
    data=shipments_df, x="MEDISPAN_SHORT_LBL_NM", y="TAT", ax=ax, palette="Set2"
)
ax.set_title("TAT by Drug Product", fontweight="bold")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# Profile TAT outliers (> 95th percentile)
tat_outliers = shipments_df[shipments_df["TAT"] > p95]
display(Markdown(f"**TAT Outlier Profile (TAT > {p95:.2f}, n={len(tat_outliers)})**"))
if len(tat_outliers) > 0:
    print("Drug distribution:")
    print(tat_outliers["MEDISPAN_SHORT_LBL_NM"].value_counts().to_string())
    print(f"\nPayer type distribution:")
    print(tat_outliers["PAYR_TYPE"].value_counts().to_string())
    print(f"\nPatient state distribution:")
    print(tat_outliers["PATIENT_ST_CD"].value_counts().to_string())
    print(f"\nHub indicator distribution:")
    print(tat_outliers["HUB_IND_ID"].value_counts().to_string())
    print(f"\nPatient type distribution:")
    print(tat_outliers["PATIENT_TYPE"].value_counts().to_string())

In [ ]:
# --- GAP_DAYS Persistence / Adherence Analysis ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GAP_DAYS distribution by patient type
ax = axes[0]
for pt, color in zip(["N", "R"], ["#4C72B0", "#DD8452"]):
    subset = shipments_df[shipments_df["PATIENT_TYPE"] == pt]["GAP_DAYS"]
    subset.hist(
        bins=20,
        ax=ax,
        alpha=0.6,
        label=f"{pt} (n={len(subset)})",
        edgecolor="black",
        color=color,
    )
ax.axvline(
    14,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Adherence risk threshold (14 days)",
)
ax.set_title("GAP_DAYS Distribution by Patient Type", fontweight="bold")
ax.set_xlabel("GAP_DAYS")
ax.legend(fontsize=8)

# Adherence risk segmentation
adherence_risk = shipments_df["GAP_DAYS"] > 14
ax = axes[1]
risk_by_type = (
    pd.crosstab(shipments_df["PATIENT_TYPE"], adherence_risk, normalize="index") * 100
)
risk_by_type.columns = ["Adherent (≤14d)", "At-risk (>14d)"]
risk_by_type.plot.bar(
    ax=ax, stacked=True, color=["#55A868", "#C44E52"], edgecolor="black"
)
ax.set_title("Adherence Risk by Patient Type (%)", fontweight="bold")
ax.set_ylabel("Percentage")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=9)

# At-risk profile
ax = axes[2]
at_risk = shipments_df[adherence_risk]
if len(at_risk) > 0:
    at_risk["THRPC_CLASS"].value_counts().plot.barh(
        ax=ax, color=sns.color_palette("Set2"), edgecolor="black"
    )
    ax.set_title(
        f"At-Risk Patients (GAP>14d, n={len(at_risk)}) by Therapeutic Class",
        fontweight="bold",
    )
    ax.set_xlabel("Count")

plt.tight_layout()
plt.show()

print(
    f"Records with GAP_DAYS > 14: {adherence_risk.sum()} ({adherence_risk.mean()*100:.1f}%)"
)
print(
    f"New patients with GAP_DAYS > 0: {((shipments_df['PATIENT_TYPE']=='N') & (shipments_df['GAP_DAYS']>0)).sum()}"
)

In [ ]:
# --- Fulfillment Flag Analysis (SMF_FLAG_ID × HUB_IND_ID) ---

# Crosstab
ct_flags = pd.crosstab(
    shipments_df["SMF_FLAG_ID"], shipments_df["HUB_IND_ID"], margins=True
)
display(Markdown("**SMF_FLAG_ID × HUB_IND_ID Crosstab**"))
display(ct_flags)

# TAT and GAP_DAYS by flag combinations
shipments_df["FLAG_COMBO"] = (
    "SMF="
    + shipments_df["SMF_FLAG_ID"].astype(str)
    + " / HUB="
    + shipments_df["HUB_IND_ID"].astype(str)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=shipments_df, x="FLAG_COMBO", y="TAT", ax=axes[0], palette="Set2")
axes[0].set_title("TAT by Fulfillment Flag Combination", fontweight="bold")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(data=shipments_df, x="FLAG_COMBO", y="GAP_DAYS", ax=axes[1], palette="Set3")
axes[1].set_title("GAP_DAYS by Fulfillment Flag Combination", fontweight="bold")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# Summary stats per combo
display(Markdown("**Median TAT & GAP_DAYS by Flag Combination**"))
display(
    shipments_df.groupby("FLAG_COMBO")[["TAT", "GAP_DAYS"]]
    .agg(["median", "mean", "count"])
    .round(2)
)

In [ ]:
# --- Hub vs Non-Hub Processing Comparison ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

compare_metrics = ["TAT", "GAP_DAYS", "ASST_OBTAIN_VAL"]
for i, col in enumerate(compare_metrics):
    sns.boxplot(
        data=shipments_df,
        x="HUB_IND_ID",
        y=col,
        ax=axes[i],
        palette={0: "#C44E52", 1: "#55A868"},
    )
    axes[i].set_title(f"{col}: Hub (1) vs Non-Hub (0)", fontweight="bold")
    axes[i].set_xlabel("HUB_IND_ID")

plt.tight_layout()
plt.show()

# Mann-Whitney U test for each metric
display(Markdown("**Mann-Whitney U Tests: Hub vs Non-Hub**"))
hub = shipments_df[shipments_df["HUB_IND_ID"] == 1]
non_hub = shipments_df[shipments_df["HUB_IND_ID"] == 0]

for col in compare_metrics:
    if len(hub) > 0 and len(non_hub) > 0:
        stat, pval = stats.mannwhitneyu(hub[col], non_hub[col], alternative="two-sided")
        sig = (
            "***"
            if pval < 0.001
            else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
        )
        print(
            f"  {col}: U={stat:.0f}, p={pval:.4f} {sig}  "
            f"(Hub median={hub[col].median():.2f}, Non-hub median={non_hub[col].median():.2f})"
        )

In [ ]:
# --- 30-Day Supply Compliance & Adjusted Days of Supply ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RX30_DAYS_SUPPLY vs actual SHIP_QTY
ax = axes[0]
sns.boxplot(
    data=shipments_df, x="RX30_DAYS_SUPPLY", y="SHIP_QTY", ax=ax, palette="Set2"
)
ax.set_title("SHIP_QTY by RX30_DAYS_SUPPLY Flag", fontweight="bold")
ax.set_xlabel("RX30_DAYS_SUPPLY (1=standard 30-day)")

# RX30_DAYS_SUPPLY vs ADJ_DAYS_SPPL
ax = axes[1]
sns.boxplot(
    data=shipments_df, x="RX30_DAYS_SUPPLY", y="ADJ_DAYS_SPPL", ax=ax, palette="Set2"
)
ax.set_title("ADJ_DAYS_SPPL by RX30_DAYS_SUPPLY Flag", fontweight="bold")
ax.set_xlabel("RX30_DAYS_SUPPLY (1=standard 30-day)")

# Scatter: SHIP_QTY vs ADJ_DAYS_SPPL
ax = axes[2]
colors = shipments_df["RX30_DAYS_SUPPLY"].map({0: "#C44E52", 1: "#4C72B0"})
ax.scatter(
    shipments_df["SHIP_QTY"],
    shipments_df["ADJ_DAYS_SPPL"],
    c=colors,
    alpha=0.6,
    edgecolors="black",
    s=40,
)
ax.set_title("SHIP_QTY vs ADJ_DAYS_SPPL", fontweight="bold")
ax.set_xlabel("SHIP_QTY")
ax.set_ylabel("ADJ_DAYS_SPPL")
# Legend
import matplotlib.patches as mpatches

legend_handles = [
    mpatches.Patch(color="#4C72B0", label="RX30=1"),
    mpatches.Patch(color="#C44E52", label="RX30=0"),
]
ax.legend(handles=legend_handles, fontsize=9)

plt.tight_layout()
plt.show()

# Flag mismatches: RX30_DAYS_SUPPLY=1 but ADJ_DAYS_SPPL not near 30
rx30_flagged = shipments_df[shipments_df["RX30_DAYS_SUPPLY"] == 1]
mismatch_mask = ~rx30_flagged["ADJ_DAYS_SPPL"].between(25, 35)
mismatches = rx30_flagged[mismatch_mask]
print(f"RX30_DAYS_SUPPLY=1 records: {len(rx30_flagged)}")
print(
    f"  of which ADJ_DAYS_SPPL NOT in [25-35]: {len(mismatches)} "
    f"({len(mismatches)/len(rx30_flagged)*100:.1f}%)"
)
if len(mismatches) > 0:
    print(
        f"  Mismatch ADJ_DAYS_SPPL range: {mismatches['ADJ_DAYS_SPPL'].min()} – {mismatches['ADJ_DAYS_SPPL'].max()}"
    )

# ADJ_DAYS_SPPL detailed distribution
display(Markdown("**ADJ_DAYS_SPPL Distribution**"))
print(shipments_df["ADJ_DAYS_SPPL"].describe().round(2).to_string())
print(
    f"\nCorrelation SHIP_QTY ↔ ADJ_DAYS_SPPL: {shipments_df['SHIP_QTY'].corr(shipments_df['ADJ_DAYS_SPPL']):.3f}"
)

## Phase 7: Multivariate Summary & Key Findings

Pair-plot overview of key operational metrics and consolidated summary of data insights.

In [ ]:
# --- Pair Plot: Key Operational Metrics ---

pair_cols = [
    "TAT",
    "GAP_DAYS",
    "ADJ_DAYS_SPPL",
    "SHIP_QTY",
    "ASST_OBTAIN_VAL",
    "PATIENT_TYPE",
]
g = sns.pairplot(
    shipments_df[pair_cols],
    hue="PATIENT_TYPE",
    palette={"N": "#4C72B0", "R": "#DD8452"},
    diag_kind="kde",
    plot_kws={"alpha": 0.5, "s": 30, "edgecolor": "black"},
    height=2.2,
    aspect=1.1,
)
g.figure.suptitle(
    "Pair Plot — Key Operational Metrics (colored by Patient Type)",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.show()

In [ ]:
# --- Automated Key Findings Summary ---

findings = []

# 1. Data completeness
null_total = shipments_df.isnull().sum().sum()
findings.append(
    f"1. **Data completeness**: {'No missing values detected — dataset is fully populated.' if null_total == 0 else f'{null_total} total null values found across the dataset.'}"
)

# 2. Patient mix
n_pct = (shipments_df["PATIENT_TYPE"] == "N").mean() * 100
r_pct = 100 - n_pct
findings.append(
    f"2. **Patient mix**: {r_pct:.0f}% Refill vs {n_pct:.0f}% New patients."
)

# 3. TAT
findings.append(
    f"3. **Turnaround time**: Median TAT = {shipments_df['TAT'].median():.2f} days, "
    f"Mean = {shipments_df['TAT'].mean():.2f} days. "
    f"95th percentile at {shipments_df['TAT'].quantile(0.95):.2f} days."
)

# 4. Adherence / gap days
gap_risk_pct = (shipments_df["GAP_DAYS"] > 14).mean() * 100
findings.append(
    f"4. **Adherence risk**: {gap_risk_pct:.1f}% of records have GAP_DAYS > 14 days, "
    f"indicating potential refill delay."
)

# 5. Financial assistance
assist_pct = (shipments_df["ASST_OBTAIN_VAL"] > 0).mean() * 100
findings.append(
    f"5. **Financial assistance**: {assist_pct:.1f}% of shipments received financial assistance "
    f"(mean value when provided: ${shipments_df.loc[shipments_df['ASST_OBTAIN_VAL'] > 0, 'ASST_OBTAIN_VAL'].mean():.0f})."
)

# 6. Hub processing
hub_pct = shipments_df["HUB_IND_ID"].mean() * 100
findings.append(
    f"6. **Hub processing**: {hub_pct:.1f}% of shipments routed through a patient support hub."
)

# 7. Payer mix
top_payer_type = shipments_df["PAYR_TYPE"].value_counts().idxmax()
top_payer_pct = shipments_df["PAYR_TYPE"].value_counts(normalize=True).max() * 100
findings.append(
    f"7. **Payer mix**: Dominant payer type is '{top_payer_type}' ({top_payer_pct:.1f}%)."
)

# 8. Drug portfolio
top_tc = shipments_df["THRPC_CLASS"].value_counts().idxmax()
top_tc_pct = shipments_df["THRPC_CLASS"].value_counts(normalize=True).max() * 100
findings.append(
    f"8. **Therapeutic focus**: '{top_tc}' accounts for {top_tc_pct:.1f}% of shipments."
)

# 9. Supply compliance
rx30_match = rx30_flagged["ADJ_DAYS_SPPL"].between(25, 35).mean() * 100
findings.append(
    f"9. **30-day supply compliance**: {rx30_match:.1f}% of RX30-flagged records have "
    f"ADJ_DAYS_SPPL within 25–35 day window."
)

# 10. SHIP_QTY ↔ ADJ_DAYS_SPPL correlation
corr_val = shipments_df["SHIP_QTY"].corr(shipments_df["ADJ_DAYS_SPPL"])
findings.append(
    f"10. **Quantity-supply alignment**: SHIP_QTY ↔ ADJ_DAYS_SPPL correlation = {corr_val:.3f}."
)

display(Markdown("## Key Findings\n\n" + "\n".join(findings)))